# ConvIR — Dual Degradation Training
**Rain + Fog/Haze combined restoration**

### Before running:
1. Runtime → Change runtime type → **GPU (T4)**
2. Upload your `Image_dual_degradation/` folder to Drive (or let Cell 2 clone ConvIR and you upload just the task folder)
3. Run cells **top to bottom** on first session
4. On reconnect after disconnect: run **Cell 1 → 2 → 4 → 6** — training auto-resumes

In [ ]:
# ── Cell 1: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted.')

In [ ]:
# ── Cell 2: Configuration — EDIT THESE PATHS ─────────────────────────────────
import os

# Dataset paths (from the shared Drive folder)
# Drive folder ID: 14fAQiN661u-9u-vRzjtE0CmG0ZPtamm5
# After adding shortcut to My Drive → Dataset/
TRAIN_DIR = '/content/drive/MyDrive/Dataset/Training_Dataset'
TEST_DIR  = '/content/drive/MyDrive/Dataset/Test_Dataset'

# Where your Image_dual_degradation folder lives on Drive
# Upload the whole folder there after local testing
TASK_ON_DRIVE = '/content/drive/MyDrive/ConvIR/Image_dual_degradation'

# Checkpoint directory on Drive (auto-created)
DRIVE_CKPT = '/content/drive/MyDrive/ConvIR_checkpoints/dual_degradation'

# Local working copy (Colab runtime — fast I/O, lost on disconnect)
LOCAL_TASK = '/content/Image_dual_degradation'
LOCAL_CKPT = f'{LOCAL_TASK}/checkpoints'

# Training config
MODEL_SIZE  = 'S'    # S=small(3 blocks)  B=base(7)  L=large(15)
BATCH_SIZE  = 8
PATCH_SIZE  = 256
NUM_EPOCHS  = 200
LR          = 1e-4
LR_MIN      = 1e-6
SAVE_EVERY  = 5      # save to Drive every N epochs
NUM_WORKERS = 2      # keep low on Colab

os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(LOCAL_CKPT, exist_ok=True)
print('Config OK')
print(f'  Train dir   : {TRAIN_DIR}')
print(f'  Drive ckpts : {DRIVE_CKPT}')
print(f'  Model size  : ConvIR-{MODEL_SIZE}')

In [ ]:
# ── Cell 3: Install dependencies + clone ConvIR ───────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print(result.stdout[-500:] if result.stdout.strip() else 'OK')

print('Installing packages...')
run('pip install -q einops scikit-image pytorch_msssim opencv-python-headless')

print('Cloning ConvIR repo...')
if not os.path.isdir('/content/ConvIR'):
    run('git clone https://github.com/c-yn/ConvIR.git /content/ConvIR')
else:
    print('  Already cloned.')

print('Installing warmup scheduler...')
run('cd /content/ConvIR/pytorch-gradual-warmup-lr && pip install -q -e .')

print('Done.')

In [ ]:
# ── Cell 4: Copy task folder to local runtime ─────────────────────────────────
# We copy from Drive to /content/ for fast local I/O during training.
# The models/ folder is taken from the ConvIR repo (Motion_Deblurring/models/)
import shutil

# Copy your Image_dual_degradation folder from Drive → local
if os.path.isdir(TASK_ON_DRIVE):
    if os.path.isdir(LOCAL_TASK):
        shutil.rmtree(LOCAL_TASK)
    shutil.copytree(TASK_ON_DRIVE, LOCAL_TASK)
    print(f'Copied task folder from Drive to {LOCAL_TASK}')
else:
    print(f'ERROR: Task folder not found on Drive: {TASK_ON_DRIVE}')
    print('Upload your Image_dual_degradation/ folder to that location first.')

# Copy models/ from the cloned ConvIR repo if not already present
models_src = '/content/ConvIR/Motion_Deblurring/models'
models_dst = f'{LOCAL_TASK}/models'
if not os.path.isdir(models_dst) and os.path.isdir(models_src):
    shutil.copytree(models_src, models_dst)
    print(f'Copied models/ from repo')
elif os.path.isdir(models_dst):
    print(f'models/ already present')
else:
    print(f'ERROR: Could not find models/ at {models_src}')

os.makedirs(LOCAL_CKPT, exist_ok=True)
print(f'\nTask folder contents:')
for f in sorted(os.listdir(LOCAL_TASK)):
    print(f'  {f}')

In [ ]:
# ── Cell 5: Verify dataset structure ─────────────────────────────────────────
# Run this to confirm the dataset path is correct and pairs are found.
os.chdir(LOCAL_TASK)
!python verify_dataset.py --root {TRAIN_DIR} --batch_size 2

In [ ]:
# ── Cell 6: TRAIN ─────────────────────────────────────────────────────────────
# Auto-resumes from latest checkpoint in Drive on reconnect.
# Checkpoints are written to Drive every SAVE_EVERY epochs.

os.chdir(LOCAL_TASK)

cmd = [
    sys.executable, 'train.py',
    '--train_dir',   TRAIN_DIR,
    '--local_ckpt',  LOCAL_CKPT,
    '--drive_ckpt',  DRIVE_CKPT,
    '--model_size',  MODEL_SIZE,
    '--batch_size',  str(BATCH_SIZE),
    '--patch_size',  str(PATCH_SIZE),
    '--num_epochs',  str(NUM_EPOCHS),
    '--lr',          str(LR),
    '--lr_min',      str(LR_MIN),
    '--num_workers', str(NUM_WORKERS),
    '--save_every',  str(SAVE_EVERY),
]

print('Command:', ' '.join(cmd))
print('-' * 60)

import subprocess
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExited with code {proc.returncode}')

In [ ]:
# ── Cell 7: Manual Drive backup ───────────────────────────────────────────────
# Run any time to force-sync all local checkpoints to Drive.
import glob, shutil

ckpt_files = glob.glob(f'{LOCAL_CKPT}/*.pth')
if not ckpt_files:
    print('No local checkpoints found yet.')
else:
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    for src in sorted(ckpt_files):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f'  Backed up: {os.path.basename(src)}')
    print(f'Done → {DRIVE_CKPT}')

In [ ]:
# ── Cell 8: EVALUATE ─────────────────────────────────────────────────────────
import glob

os.chdir(LOCAL_TASK)

# Pick best_model.pth, fall back to latest epoch checkpoint
best_ckpt = os.path.join(DRIVE_CKPT, 'best_model.pth')
if not os.path.exists(best_ckpt):
    ckpts = sorted(glob.glob(f'{DRIVE_CKPT}/epoch_*.pth'))
    best_ckpt = ckpts[-1] if ckpts else ''

if not best_ckpt:
    print('No checkpoint found. Run Cell 6 first.')
else:
    print(f'Evaluating: {best_ckpt}')
    cmd = [
        sys.executable, 'test.py',
        '--test_dir',   TEST_DIR,
        '--model_path', best_ckpt,
        '--model_size', MODEL_SIZE,
        '--save_images',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

In [ ]:
# ── Cell 9: List Drive checkpoints ───────────────────────────────────────────
import glob, torch

ckpts = sorted(glob.glob(f'{DRIVE_CKPT}/*.pth'))
if not ckpts:
    print('No checkpoints on Drive yet.')
else:
    print(f'Checkpoints in {DRIVE_CKPT}:')
    print(f'  {"File":<28} {"Epoch":>6} {"Loss":>8} {"Size":>8}')
    print('  ' + '-'*54)
    for c in ckpts:
        size_mb = os.path.getsize(c) / 1e6
        try:
            ck = torch.load(c, map_location='cpu')
            ep   = ck.get('epoch', '?')  if isinstance(ck, dict) else '?'
            loss = ck.get('loss',  '?')  if isinstance(ck, dict) else '?'
            loss_s = f'{loss:.5f}' if isinstance(loss, float) else str(loss)
        except Exception:
            ep, loss_s = '?', '?'
        print(f'  {os.path.basename(c):<28} {str(ep):>6} {loss_s:>8} {size_mb:>6.1f}MB')